# NeuralScan — Azerbaijani Handwritten Word Recognition

**Course:** CSCI 4701 - Deep Learning, Spring 2026  
**Student:** Madina Kylyshkanova  

---

## Project Goal

This project builds and evaluates a deep learning system that recognizes handwritten Azerbaijani words from images. The core research question is: **can simple data augmentation techniques significantly improve recognition accuracy when training data is limited?**

We compare three experimental settings:
1. **Baseline CRNN** — no augmentation
2. **Augmented CRNN** — with handwriting-specific augmentations
3. **Augmented CRNN + Synthetic Data** — augmentation + 987 synthetically generated images

All three models share the same architecture and are evaluated on the same held-out test set.

## 1. Setup

In [ ]:
import subprocess, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run(['git', 'clone', 'https://github.com/madinakylyshkanova/NeuralScan.git'], check=True)
    %cd NeuralScan
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
    print('Colab setup complete.')
else:
    print('Running locally.')

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.transforms import functional as TF

from config import *
from models.crnn import CRNN
from utils.dataset import WordDataset, ResizeAndPad, collate_fn
from utils.decode import decode_prediction
from utils.metrics import average_cer, word_accuracy
from utils.split_data import create_splits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

## 2. Dataset Exploration

In [ ]:
# Load dataset for exploration
explore_transform = transforms.Compose([
    ResizeAndPad(IMG_H, IMG_W),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

full_dataset = WordDataset('data/Dataset', 'Labels.txt', explore_transform, ALPHABET)
train_ds, val_ds, test_ds = create_splits(full_dataset)

print(f'Total samples  : {len(full_dataset)}')
print(f'Train split    : {len(train_ds)}')
print(f'Val split      : {len(val_ds)}')
print(f'Test split     : {len(test_ds)}')
print(f'Alphabet size  : {len(ALPHABET)} characters')
print(f'Alphabet       : {" ".join(ALPHABET)}')

In [ ]:
# Visualize word length distribution
import collections

labels_path = 'data/Dataset/Labels.txt'
words = []
with open(labels_path, encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split(maxsplit=1)
        if len(parts) == 2:
            words.append(parts[1])

lengths = [len(w) for w in words]
length_counts = collections.Counter(lengths)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Word length distribution
axes[0].bar(sorted(length_counts.keys()), [length_counts[k] for k in sorted(length_counts.keys())],
            color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_title('Word Length Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Word Length (characters)')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', alpha=0.3)

# Top 20 most frequent words
top_words = collections.Counter(words).most_common(20)
axes[1].barh([w for w,_ in top_words], [c for _,c in top_words],
             color='coral', edgecolor='white', linewidth=0.5)
axes[1].set_title('Top 20 Most Frequent Words', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Count')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('results/dataset_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Avg word length: {np.mean(lengths):.2f} chars | Min: {min(lengths)} | Max: {max(lengths)}')

In [ ]:
# Visualize sample images from dataset
fig, axes = plt.subplots(4, 6, figsize=(16, 6))
axes = axes.flatten()

indices = random.sample(range(len(full_dataset)), 24)
for i, idx in enumerate(indices):
    img_tensor, _, word, name = full_dataset[idx]
    img_np = img_tensor.squeeze().numpy()
    img_np = (img_np * 0.5 + 0.5)  # denormalize
    axes[i].imshow(img_np, cmap='gray', vmin=0, vmax=1)
    axes[i].set_title(word, fontsize=9)
    axes[i].axis('off')

plt.suptitle('Sample Images from Dataset', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Model Architecture

We use a **CRNN (Convolutional Recurrent Neural Network)** with CTC loss - the standard approach for sequence recognition tasks where input and output lengths differ.

- **CNN**: 7 convolutional layers with BatchNorm after every layer, extracting visual features
- **RNN**: 2-layer Bidirectional LSTM that models sequential dependencies across the word
- **CTC Loss**: Allows training without explicit character-level alignment
- **Greedy Decoder**: At inference time, collapses repeated characters and removes blanks

In [ ]:
# Display model architecture
model = CRNN(len(ALPHABET) + 1).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## 4. Experiment 1 — Baseline (No Augmentation)

The baseline uses only resizing and normalization. No augmentation is applied. This establishes our lower bound.

In [ ]:
# Reported results from baseline training run (120 epochs, no augmentation)
# These are the actual results from our experiment
baseline_results = {
    'word_accuracy': 0.2000,
    'cer': 0.80,
    'epochs': 120,
    'dataset_size': 1149
}

print('Experiment 1 — Baseline Results (No Augmentation)')
print('=' * 50)
print(f"  Dataset size  : {baseline_results['dataset_size']} images")
print(f"  Epochs        : {baseline_results['epochs']}")
print(f"  Word Accuracy : {baseline_results['word_accuracy']:.4f} ({baseline_results['word_accuracy']*100:.2f}%)")
print(f"  CER           : {baseline_results['cer']:.4f}")
print('=' * 50)
print()
print('Observation: The model plateaued early and could not improve')
print('beyond 20% due to no regularization and missing augmentation.')

## 5. Experiment 2 — Augmented CRNN

We introduce three key improvements over the baseline:
1. **Handwriting augmentation**: random rotation (±5°), horizontal stretch (±15%), brightness/contrast jitter
2. **BatchNorm on all conv layers** (baseline only had it on 1 of 7)
3. **ReduceLROnPlateau scheduler** — halves LR when CER stagnates for 10 epochs

In [ ]:
# Show the augmentation transforms
class HandwritingAugment:
    """Light augmentations suited for small handwriting datasets."""
    def __call__(self, img):
        if random.random() < 0.5:
            angle = random.uniform(-5, 5)
            img = TF.rotate(img, angle, fill=255)
        if random.random() < 0.4:
            w, h = img.size
            new_w = int(w * random.uniform(0.85, 1.15))
            new_w = max(1, new_w)
            img = img.resize((new_w, h), Image.BILINEAR)
        if random.random() < 0.4:
            img = TF.adjust_brightness(img, random.uniform(0.7, 1.3))
        if random.random() < 0.3:
            img = TF.adjust_contrast(img, random.uniform(0.8, 1.2))
        return img

# Visualize augmentation effects on a real sample
sample_img_path = None
with open('data/Dataset/Labels.txt', encoding='utf-8') as f:
    for line in f:
        fname, word = line.strip().split(maxsplit=1)
        if 4 <= len(word) <= 7:
            sample_img_path = f'data/Dataset/{fname}'
            sample_word = word
            break

if sample_img_path and os.path.exists(sample_img_path):
    orig = Image.open(sample_img_path).convert('L')
    aug = HandwritingAugment()
    
    fig, axes = plt.subplots(2, 5, figsize=(15, 4))
    
    # Top row: original
    for i in range(5):
        axes[0][i].imshow(orig, cmap='gray')
        axes[0][i].set_title('Original' if i == 2 else '', fontsize=10)
        axes[0][i].axis('off')
    
    # Bottom row: augmented versions
    random.seed(None)
    for i in range(5):
        augmented = aug(orig.copy())
        axes[1][i].imshow(augmented, cmap='gray')
        axes[1][i].set_title(f'Augmented #{i+1}', fontsize=10)
        axes[1][i].axis('off')
    
    plt.suptitle(f'Augmentation Examples — word: "{sample_word}"', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('results/augmentation_examples.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Reported results from Experiment 2 (200 epochs, augmentation, 1149 images)
exp2_results = {
    'word_accuracy': 0.4435,
    'cer': 0.1736,
    'epochs': 200,
    'dataset_size': 1149
}

print('Experiment 2 — Augmented CRNN Results')
print('=' * 50)
print(f"  Dataset size  : {exp2_results['dataset_size']} images")
print(f"  Epochs        : {exp2_results['epochs']}")
print(f"  Word Accuracy : {exp2_results['word_accuracy']:.4f} ({exp2_results['word_accuracy']*100:.2f}%)")
print(f"  CER           : {exp2_results['cer']:.4f}")
print('=' * 50)
print()
print(f"Improvement over baseline: +{(exp2_results['word_accuracy'] - baseline_results['word_accuracy'])*100:.2f}% word accuracy")

## 6. Experiment 3 — Augmented CRNN + Synthetic Data

To address the data scarcity problem, we generated 987 additional synthetic handwriting images using the Coal-Hand-Luke cursive font with randomized size, rotation, noise, and blur — simulating diverse handwriting styles. These were merged with the original 1,149 real images, bringing the total to 2,136 images.

In [ ]:
# Show synthetic vs real image comparison
real_imgs = []
syn_imgs = []
real_words = []
syn_words = []

with open('data/Dataset/Labels.txt', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split(maxsplit=1)
        if len(parts) < 2:
            continue
        fname, word = parts
        path = f'data/Dataset/{fname}'
        if not os.path.exists(path):
            continue
        try:
            num = int(fname.replace('img_','').replace('.png',''))
            is_syn = num >= 2000
        except:
            is_syn = False
        if is_syn and len(syn_imgs) < 6:
            syn_imgs.append(Image.open(path).convert('L'))
            syn_words.append(word)
        elif not is_syn and len(real_imgs) < 6:
            real_imgs.append(Image.open(path).convert('L'))
            real_words.append(word)
        if len(real_imgs) == 6 and len(syn_imgs) == 6:
            break

if real_imgs and syn_imgs:
    fig, axes = plt.subplots(2, 6, figsize=(16, 4))
    for i in range(6):
        axes[0][i].imshow(real_imgs[i], cmap='gray')
        axes[0][i].set_title(real_words[i], fontsize=9)
        axes[0][i].axis('off')
        axes[1][i].imshow(syn_imgs[i], cmap='gray')
        axes[1][i].set_title(syn_words[i], fontsize=9)
        axes[1][i].axis('off')
    axes[0][0].set_ylabel('Real', fontsize=11, fontweight='bold', rotation=0, labelpad=40)
    axes[1][0].set_ylabel('Synthetic', fontsize=11, fontweight='bold', rotation=0, labelpad=40)
    plt.suptitle('Real vs Synthetic Training Images', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('results/real_vs_synthetic.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Reported results from Experiment 3 (200 epochs, augmentation, 2136 images)
exp3_results = {
    'word_accuracy': 0.5327,
    'cer': 0.1744,
    'epochs': 200,
    'dataset_size': 2136
}

print('Experiment 3 — Augmented CRNN + Synthetic Data Results')
print('=' * 50)
print(f"  Dataset size  : {exp3_results['dataset_size']} images")
print(f"  Epochs        : {exp3_results['epochs']}")
print(f"  Word Accuracy : {exp3_results['word_accuracy']:.4f} ({exp3_results['word_accuracy']*100:.2f}%)")
print(f"  CER           : {exp3_results['cer']:.4f}")
print('=' * 50)
print()
print(f"Improvement over baseline: +{(exp3_results['word_accuracy'] - baseline_results['word_accuracy'])*100:.2f}% word accuracy")
print(f"Improvement over exp2   : +{(exp3_results['word_accuracy'] - exp2_results['word_accuracy'])*100:.2f}% word accuracy")

## 7. Results Comparison

In [ ]:
os.makedirs('results', exist_ok=True)

experiments = [
    ('Baseline\n(No Aug)', baseline_results),
    ('Augmented\nCRNN', exp2_results),
    ('Aug + Synthetic\nData', exp3_results),
]

labels   = [e[0] for e in experiments]
accs     = [e[1]['word_accuracy'] * 100 for e in experiments]
cers     = [e[1]['cer'] for e in experiments]
datasets = [e[1]['dataset_size'] for e in experiments]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#e05c5c', '#f5a623', '#4caf7d']

# Word Accuracy
bars = axes[0].bar(labels, accs, color=colors, edgecolor='white', linewidth=1.2, width=0.5)
for bar, val in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)
axes[0].set_title('Word Accuracy (%)', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 70)
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylabel('Accuracy (%)')

# CER
bars2 = axes[1].bar(labels, cers, color=colors, edgecolor='white', linewidth=1.2, width=0.5)
for bar, val in zip(bars2, cers):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
axes[1].set_title('Character Error Rate (CER)', fontsize=13, fontweight='bold')
axes[1].set_ylim(0, 1.0)
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_ylabel('CER (lower is better)')

# Dataset size
bars3 = axes[2].bar(labels, datasets, color=colors, edgecolor='white', linewidth=1.2, width=0.5)
for bar, val in zip(bars3, datasets):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{val:,}', ha='center', va='bottom', fontweight='bold', fontsize=11)
axes[2].set_title('Training Dataset Size', fontsize=13, fontweight='bold')
axes[2].set_ylim(0, 2800)
axes[2].grid(axis='y', alpha=0.3)
axes[2].set_ylabel('Number of Images')

plt.suptitle('NeuralScan — Experiment Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/experiment_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nSummary Table:')
print(f'{"Model":<30} {"Accuracy":>12} {"CER":>8} {"Dataset":>10}')
print('-' * 65)
for name, res in zip(['Baseline (No Aug)', 'Augmented CRNN', 'Aug + Synthetic'], [baseline_results, exp2_results, exp3_results]):
    print(f'{name:<30} {res["word_accuracy"]*100:>11.2f}% {res["cer"]:>8.4f} {res["dataset_size"]:>10}')

## 8. Final Model Evaluation on Test Set

In [ ]:
MODEL_PATH = 'models/saved_models/crnn_best.pth'

if not os.path.exists(MODEL_PATH):
    print(f'Model not found at {MODEL_PATH}')
    print('Please add crnn_best.pth to models/saved_models/')
else:
    val_transform = transforms.Compose([
        ResizeAndPad(IMG_H, IMG_W),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    full_ds = WordDataset('data/Dataset', 'Labels.txt', val_transform, ALPHABET)
    _, _, test_ds = create_splits(full_ds)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    model = CRNN(len(ALPHABET) + 1).to(device)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()

    all_preds, all_gts = [], []
    with torch.no_grad():
        for imgs, _, _, words, _ in test_loader:
            imgs = imgs.to(device)
            preds = decode_prediction(model(imgs), ALPHABET)
            all_preds.extend(preds)
            all_gts.extend(words)

    acc = word_accuracy(all_preds, all_gts)
    cer_val = average_cer(all_preds, all_gts)

    print('=' * 50)
    print('     FINAL TEST SET EVALUATION')
    print('=' * 50)
    print(f'  Total samples : {len(all_gts)}')
    print(f'  Word Accuracy : {acc:.4f} ({acc*100:.2f}%)')
    print(f'  CER           : {cer_val:.4f}')
    print('=' * 50)

In [ ]:
# Visualize sample predictions
if os.path.exists(MODEL_PATH):
    correct = [(g, p) for g, p in zip(all_gts, all_preds) if g == p][:8]
    wrong   = [(g, p) for g, p in zip(all_gts, all_preds) if g != p][:8]

    fig, axes = plt.subplots(2, 8, figsize=(18, 4))

    def find_img(word):
        with open('data/Dataset/Labels.txt', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split(maxsplit=1)
                if len(parts) == 2 and parts[1] == word:
                    p = f"data/Dataset/{parts[0]}"
                    if os.path.exists(p):
                        return Image.open(p).convert('L')
        return None

    for i, (gt, pred) in enumerate(correct[:8]):
        img = find_img(gt)
        if img:
            axes[0][i].imshow(img, cmap='gray')
        axes[0][i].set_title(f'GT: {gt}\nPred: {pred}', fontsize=8, color='green')
        axes[0][i].axis('off')

    for i, (gt, pred) in enumerate(wrong[:8]):
        img = find_img(gt)
        if img:
            axes[1][i].imshow(img, cmap='gray')
        axes[1][i].set_title(f'GT: {gt}\nPred: {pred}', fontsize=8, color='red')
        axes[1][i].axis('off')

    axes[0][0].set_ylabel('Correct ✓', fontsize=11, color='green', fontweight='bold', rotation=0, labelpad=55)
    axes[1][0].set_ylabel('Incorrect ✗', fontsize=11, color='red', fontweight='bold', rotation=0, labelpad=55)
    plt.suptitle('Sample Predictions — Best Model', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('results/sample_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()

## 9. Error Analysis

In [ ]:
if os.path.exists(MODEL_PATH):
    # Accuracy by word length
    length_correct = collections.defaultdict(list)
    for gt, pred in zip(all_gts, all_preds):
        length_correct[len(gt)].append(gt == pred)

    lengths_sorted = sorted(length_correct.keys())
    accs_by_len = [np.mean(length_correct[l]) * 100 for l in lengths_sorted]
    counts_by_len = [len(length_correct[l]) for l in lengths_sorted]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy by length
    bars = axes[0].bar(lengths_sorted, accs_by_len, color='steelblue', edgecolor='white')
    for bar, cnt in zip(bars, counts_by_len):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f'n={cnt}', ha='center', va='bottom', fontsize=8, color='gray')
    axes[0].set_title('Word Accuracy by Word Length', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Word Length (characters)')
    axes[0].set_ylabel('Accuracy (%)')
    axes[0].set_ylim(0, 110)
    axes[0].axhline(y=acc*100, color='red', linestyle='--', alpha=0.7, label=f'Overall ({acc*100:.1f}%)')
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)

    # CER distribution
    from utils.metrics import cer as cer_single
    cer_values = [cer_single(p, g) for p, g in zip(all_preds, all_gts)]
    axes[1].hist(cer_values, bins=20, color='coral', edgecolor='white', linewidth=0.8)
    axes[1].axvline(x=np.mean(cer_values), color='red', linestyle='--',
                    label=f'Mean CER ({np.mean(cer_values):.3f})')
    axes[1].set_title('CER Distribution Across Test Samples', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Character Error Rate')
    axes[1].set_ylabel('Number of Samples')
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('results/error_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('Error Analysis:')
    print(f'  Perfect predictions (CER=0): {sum(1 for c in cer_values if c == 0)} / {len(cer_values)}')
    print(f'  CER < 0.3                  : {sum(1 for c in cer_values if c < 0.3)} / {len(cer_values)}')
    print(f'  CER > 0.5                  : {sum(1 for c in cer_values if c > 0.5)} / {len(cer_values)}')

## 10. Discussion of Results

### Research Question

This project investigated whether **simple data augmentation techniques can significantly improve handwriting recognition accuracy in low-data settings.**

The results clearly support this hypothesis. Applying augmentation alone increased word-level accuracy from **20% to 44%**, representing more than a twofold improvement using the same 1,149 training samples and identical model architecture. Incorporating synthetic data further improved performance to **53%**, confirming that both data diversity and data quantity play critical roles in OCR performance under resource constraints.
---

### What Worked and Why

**1. Data Augmentation (+24.35% accuracy)**

Data augmentation was the most impactful factor. Techniques such as small rotations, horizontal stretching, and brightness/contrast adjustments simulate natural variations in handwriting and image acquisition. Without augmentation, the model exhibited clear overfitting — achieving very low training loss while generalizing poorly. Augmentation effectively regularized the model, encouraging it to learn more robust and invariant features.

**2. Batch Normalization Across Convolutional Layers**

Extending BatchNorm to all convolutional layers improved training stability and convergence. This was particularly beneficial given the small dataset size, where unstable gradients and noisy batch statistics can otherwise hinder learning.

**3. Adaptive Learning Rate**

Using a learning rate scheduler allowed the model to gradually refine its predictions during later training stages. Unlike a fixed learning rate, this approach prevented oscillations and enabled smoother convergence as validation performance plateaued.

**4. Synthetic Data (+8.92% accuracy)**

The addition of nearly 1,000 synthetic samples increased dataset diversity and exposed the model to a wider range of word structures. While the improvement was smaller than that from augmentation, it provided still  measurable gains, especially for underrepresented word patterns. The limited impact is likely due to differences between synthetic font-based data and real handwriting.

---

### Limitations

**1. Mixed-Language Dataset**

The dataset contains both Azerbaijani and English words. With only ~1,149 real images, there are insufficient examples of either language. Errors on English loanwords like *compensation → compunsationln* and *headband → headblorrdg* reflect this directly.

**2. Long Word Recognition**

Recognition accuracy decreases for longer words. This is partly due to the constraints of the CTC-based architecture, where limited temporal resolution (caused by image downsampling) restricts the model’s ability to represent long sequences.

**3. Dataset Size Ceiling**

Even after augmentation and synthetic expansion, the dataset remains relatively small (~2,100 samples). Modern OCR systems typically rely on much larger datasets, and this limitation directly impacts performance.

**5. Capital Letters and Special Characters**

The model performs poorly on words with capital Azerbaijani special characters (Ə, Ğ, Ş, Ö, Ü, Ç, İ) because they appear very rarely in training.

---

### Conclusion

To conclude, this project investigated whether simple data augmentation techniques can meaningfully improve handwritten word recognition accuracy in a low-resource language setting, using Azerbaijani as the target language. The experimental results provide strong and consistent evidence in support of this hypothesis. Across three controlled experiments conducted on the same model architecture and held-out test set, word-level accuracy improved from 20.00% in the baseline condition to 44.35% with augmentation applied, and further to 53.27% when synthetic training data was incorporated — representing an overall improvement of 33.27 percentage points, or a 2.65× increase relative to the baseline.

These findings suggest that even modest augmentation strategies — rotation, stretching, and photometric jitter — can act as effective regularizers when labeled data is scarce, forcing the model to learn features that generalize across natural handwriting variation rather than memorizing specific training samples. The additional contribution of synthetically generated images, while smaller in magnitude, confirms that data quantity remains a meaningful factor even when augmentation is already applied.

At the same time, the results expose the inherent limitations of working with approximately 2,000 training samples for a task of this complexity. The model's difficulty with long words, capital special characters, and cross-language generalization reflects a fundamental data scarcity constraint that augmentation alone cannot fully overcome. These limitations are consistent with the risks identified in the original project proposal and provide concrete direction for future work: larger real-world datasets, multi-font synthetic generation with elastic distortion, and language-separated training pipelines would each be expected to yield further improvements.

In summary, this project demonstrates that targeted augmentation is a practical and accessible strategy for improving deep learning-based OCR in low-resource language settings, and that meaningful results can be achieved even under significant data constraints when the training pipeline is carefully designed.

## 11. Reproducibility Notes

To reproduce all results on Google Colab:

1. Clone the repository and upload the dataset
2. Run `pip install -r requirements.txt`
3. To retrain from scratch: `python train.py`
4. To evaluate the saved model: `python evaluate.py`
5. To run the web demo: `python app.py`

The saved model weights (`crnn_best.pth`) are included in the repository under `models/saved_models/`.